# GoldenRetrieverLang
## Especificación Técnica Completa

**Lenguaje de programación temático académico para prácticas de compiladores e intérpretes.**

---

## Tabla de Contenidos

1. Arquitectura General
2. Componente Lexical (Lexer)
3. Componente Sintáctico (Parser)
4. Tabla de Símbolos
5. Funciones Auxiliares
6. Sistema de Compilación
7. Comentarios en el Código
8. Operaciones Aritméticas
9. Flujo de Ejecución Detallado
10. Ejemplos Complejos
11. Validaciones de Tipo

---

## 1. Arquitectura General

GoldenRetrieverLang implementa una arquitectura clásica de compilador en tres fases:

```
┌─────────────────────────────────────────────────────────────┐
│                     CÓDIGO FUENTE                           │
│                      (prueba)                               │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│         FASE 1: ANÁLISIS LÉXICO (lexer.l)                  │
│    Convierte caracteres en tokens (símbolos léxicos)       │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│      FASE 2: ANÁLISIS SINTÁCTICO (parser.y)               │
│   Verifica gramática y ejecuta (bottom-up parser LALR)     │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  FASE 3: EVALUACIÓN Y EJECUCIÓN (parser_helper.c)          │
│  Operaciones aritméticas, gestión de símbolos, output      │
└──────────────────────────┬──────────────────────────────────┘
                           │
                           ▼
┌─────────────────────────────────────────────────────────────┐
│                      SALIDA ESTÁNDAR                        │
└─────────────────────────────────────────────────────────────┘
```

---

## 2. Componente Lexical (Lexer)

### ¿Qué es?
El analizador léxico (lexical analyzer) lee el código fuente carácter por carácter y los agrupa en **tokens** (unidades léxicas significativas).

### Definiciones de Patrones (lexer.l)

```flex
LETRA      [A-Za-z_]
DIGITO     [0-9]
IDF        {LETRA}({LETRA}|{DIGITO})*
ENTERO     {DIGITO}+
CADENA     \"([^\"\\]|\\.)*\"
```

Estos son patrones usando expresiones regulares que definen qué caracteres conforman cada tipo de token.

### Ejemplo: Tokenización

**Entrada:**
```
HUESO pelotas = 12;
```

**Salida de tokens:**

| Lexema | Tipo de Token | Valor |
|--------|---------------|-------|
| `HUESO` | `HUESO` | - |
| `pelotas` | `ID` | "pelotas" |
| `=` | `ASSIGN` | - |
| `12` | `NUMBER` | 12 |
| `;` | `SEMI` | - |

### Tokens Especiales del Lenguaje

```flex
"HUESO"         { return HUESO; }   // Declaración de entero
"CORREA"        { return CORREA; }  // Declaración de string
"LADRA"         { return LADRA; }   // Impresión en pantalla

{ENTERO}         { yylval.ival = atoi(yytext); return NUMBER; }
{CADENA}         { yylval.sval = unescape_string(yytext); return STRING; }
{IDF}            { yylval.sval = xstrdup(yytext); return ID; }

"+"              { return '+'; }
"-"              { return '-'; }
"*"              { return '*'; }
"/"              { return '/'; }
"="              { return '='; }
"("              { return '('; }
")"              { return ')'; }
";"              { return ';'; }
```

### Manejo de Comentarios

```flex
"//"[^\n]*                    { /* comentario de linea, se ignora */ }
"/*"([^*]|\*+[^*/])*\*+"/" { /* comentario de bloque */ }
[ \t\r\n]+                   { /* espacios en blanco se ignoran */ }
```

---

## 3. Componente Sintáctico (Parser)

### ¿Qué es?
El análisis sintáctico verifica que los tokens sigan las reglas gramaticales del lenguaje y coordina la ejecución.

### Gramática YACC/Bison (parser.y)

```yacc
program:
    statements
;

statements:
    /* empty */
    | statements statement
;

statement:
    HUESO ID '=' expr ';'
    | CORREA ID '=' expr ';'
    | ID '=' expr ';'
    | LADRA '(' expr ')' ';'
;

expr:
    expr '+' term
    | expr '-' term
    | term
;

term:
    term '*' factor
    | term '/' factor
    | factor
;

factor:
    NUMBER
    | STRING
    | ID
    | '(' expr ')'
;
```

### Precedencia de Operadores

```yacc
%left '+' '-'      // Suma y resta: asociatividad izquierda
%left '*' '/'      // Multiplicación y división: asociatividad izquierda
```

**Ejemplo:** `2 + 3 * 4` se evalúa como `2 + (3 * 4) = 14`, **no** como `(2 + 3) * 4 = 20`

---

## 4. Tabla de Símbolos (symbols.c / symbols.h)

### ¿Qué es?
Una base de datos que almacena todas las variables declaradas durante la ejecución.

### Estructura de Datos

```c
typedef enum {
    TYPE_INT = 0,
    TYPE_STRING = 1
} ValueType;

typedef struct {
    ValueType type;          // Tipo de dato
    int int_value;           // Valor entero
    char* str_value;         // Valor string
} Value;

typedef struct {
    char name[128];          // Nombre de la variable
    int used;                // ¿Está usada?
    Value value;             // Valor actual
} Symbol;

#define MAX_SYMBOLS 256
static Symbol table[MAX_SYMBOLS];  // Máximo 256 variables
```

### Funciones Principales

**1. `declare_symbol(name, expected_type, value)`**
- Crea una nueva variable en la tabla
- Valida que no exista duplicada
- Valida que el tipo coincida

**2. `get_symbol_value(name, out_value)`**
- Busca una variable por nombre
- Retorna una copia del valor
- Error si no existe

**3. `assign_symbol(name, value)`**
- Reasigna el valor de una variable existente
- Valida que exista y que el tipo coincida

**4. `cleanup_symbols()`**
- Libera toda la memoria asignada
- Se llama al terminar la ejecución

### Ejemplo: Sesión de Símbolos

```c
// Paso 1: Declarar edad (entero)
declare_symbol("edad", TYPE_INT, make_int(5));
// Tabla: { edad: INT = 5 }

// Paso 2: Declarar nombre (string)
declare_symbol("nombre", TYPE_STRING, make_string("Max"));
// Tabla: { edad: INT = 5, nombre: STRING = "Max" }

// Paso 3: Obtener valor de edad
Value v;
get_symbol_value("edad", &v);  // v.type = TYPE_INT, v.int_value = 5

// Paso 4: Reasignar edad
assign_symbol("edad", make_int(6));
// Tabla: { edad: INT = 6, nombre: STRING = "Max" }

// Paso 5: Limpiar memoria
cleanup_symbols();  // Libera todos los valores
```

---

## 5. Funciones Auxiliares (parser_helper.c / parser_helper.h)

### ¿Qué son?
Funciones que implementan operaciones del lenguaje con validaciones de tipo y manejo de errores.

### Operaciones Aritméticas

```c
// Suma (enteros o strings)
Value add_values_checked(Value left, Value right)

// Resta (solo enteros)
Value sub_values_checked(Value left, Value right)

// Multiplicación (solo enteros)
Value mul_values_checked(Value left, Value right)

// División (solo enteros, con validación de cero)
Value div_values_checked(Value left, Value right)

// Impresión en pantalla
void print_value_checked(Value value)
```

### Ejemplos de Operaciones

**Suma de enteros:**
```c
Value v1 = make_int(3);
Value v2 = make_int(5);
Value result = add_values_checked(v1, v2);  // 8
```

**Concatenación de strings:**
```c
Value s1 = make_string("Golden");
Value s2 = make_string(" Retriever");
Value result = add_values_checked(s1, s2);  // "Golden Retriever"
```

**División entera:**
```c
Value v1 = make_int(10);
Value v2 = make_int(3);
Value result = div_values_checked(v1, v2);  // 3 (no 3.33...)
```

---

## 6. Sistema de Compilación (Makefile)

### Estructura del Makefile

```makefile
CC = gcc
FLEX = flex
BISON = bison
CFLAGS = -Wall -Wextra -std=c11
TARGET = golden

all: $(TARGET)

parser.tab.c parser.tab.h: parser.y
	$(BISON) -d parser.y

lex.yy.c: lexer.l parser.tab.h
	$(FLEX) lexer.l

$(TARGET): parser.tab.c lex.yy.c parser_helper.c symbols.c
	$(CC) $(CFLAGS) -o $(TARGET) parser.tab.c lex.yy.c parser_helper.c symbols.c

run: $(TARGET)
	./$(TARGET) < prueba

clean:
	rm -f $(TARGET) parser.tab.c parser.tab.h lex.yy.c
```

### Comandos Disponibles

| Comando | Acción |
|---------|--------|
| `make` o `make all` | Compila el programa |
| `make run` | Ejecuta el programa |
| `make clean` | Elimina archivos generados |

**Flujo de compilación:**
```
1. bison -d parser.y       → Genera parser.tab.c, parser.tab.h
2. flex lexer.l            → Genera lex.yy.c
3. gcc -o golden ...       → Compila y enlaza todo
```

---

## 7. Comentarios en el Código

GoldenRetrieverLang soporta dos tipos de comentarios que se ignoran durante el análisis léxico.

### Comentarios de Línea (`//`)

Todo lo que aparezca después de `//` hasta el final de la línea se ignora:

```
HUESO edad = 5;        // Edad del Golden Retriever
HUESO pelotas = 10;    // Cantidad de juguetes disponibles

// Esta línea entera es un comentario
// Y esta también
```

### Comentarios de Bloque (`/* ... */`)

Todo lo que esté entre `/*` y `*/` se ignora, incluso si ocupa múltiples líneas:

```
/*
  Inicialización de variables del perro
  Programa: Sistema de cuidado de Golden Retrievers
  Versión: 1.0
*/
HUESO edad = 5;

/* Comentario en una sola línea */

LADRA(edad);  /* Imprime la edad */
```

---

## 8. Operaciones Aritméticas

Las operaciones aritméticas en GoldenRetrieverLang siguen las reglas matemáticas estándar.

### SUMA (`+`)

**Enteros:**
```
HUESO a = 10;
HUESO b = 5;
HUESO suma = a + b;  // 15
```

**Strings (Concatenación):**
```
CORREA saludo = "Golden";
CORREA raza = " Retriever";
CORREA resultado = saludo + raza;  // "Golden Retriever"
```

**Validaciones:**
- ✓ `entero + entero = entero`
- ✓ `string + string = string`
- ✗ `entero + string` → Error

### RESTA (`-`)

```
HUESO edad = 10;
HUESO anos_vividos = 3;
HUESO edad_hace_3 = edad - anos_vividos;  // 7
```

**Validaciones:**
- ✓ Solo enteros: `entero - entero = entero`
- ✗ `string - string` → Error

### MULTIPLICACIÓN (`*`)

```
HUESO paseos_dia = 3;
HUESO dias = 7;
HUESO paseos_semana = paseos_dia * dias;  // 21
```

**Validaciones:**
- ✓ Solo enteros: `entero * entero = entero`

### DIVISIÓN (`/`)

**Nota:** División ENTERA (sin decimales)

```
HUESO pelotas = 20;
HUESO perros = 3;
HUESO pelotas_cada = pelotas / perros;  // 6 (no 6.67)

HUESO a = 10;
HUESO b = 3;
LADRA(a / b);  // Salida: 3
```

**Validaciones:**
- ✓ Solo enteros: `entero / entero = entero`
- ✗ División entre cero → Error

### PRECEDENCIA DE OPERADORES

Se sigue el orden matemático estándar:

```
1. Multiplicación (*) y División (/)
2. Suma (+) y Resta (-)
```

**Ejemplos:**
```
HUESO x = 2 + 3 * 4;
LADRA(x);  // Salida: 14 (no 20)
           // Porque: 2 + (3 * 4) = 2 + 12 = 14

HUESO y = 10 + 20 / 5 - 2;
LADRA(y);  // Salida: 12
           // Porque: 10 + (20 / 5) - 2 = 10 + 4 - 2 = 12
```

**Usar paréntesis para cambiar orden:**
```
HUESO a = (2 + 3) * 4;  // 20 (suma primero)
HUESO b = 2 + 3 * 4;    // 14 (multiplicación primero)
```

---

## 9. Flujo de Ejecución Detallado

### Archivo de Prueba Completo

```
// demo golden retriever
HUESO pelotas = 12;
HUESO paseos = 4;
HUESO total = pelotas + paseos * 2;

CORREA nombre = "Max";
CORREA texto = "Golden: " + nombre;

LADRA(texto);
LADRA(total);

pelotas = pelotas - 2;
LADRA(pelotas);
```

### Fase 1: Análisis Léxico

**Línea:** `HUESO pelotas = 12;`

```
Entrada:  HUESO pelotas = 12;
Salida:   [HUESO, ID(pelotas), ASSIGN, NUMBER(12), SEMI]
```

### Fase 2: Análisis Sintáctico y Ejecución

**Paso 1: Declarar pelotas**
```
Parser: statement → HUESO ID = expr ;
Evaluación: expr = 12
Acción: declare_variable_checked("pelotas", TYPE_INT, make_int(12))

Tabla de símbolos:
  pelotas: INT = 12
```

**Paso 2: Declarar paseos**
```
Parser: statement → HUESO ID = expr ;
Evaluación: expr = 4

Tabla de símbolos:
  pelotas: INT = 12
  paseos: INT = 4
```

**Paso 3: Calcular total (con precedencia)**
```
Parser: statement → HUESO ID = expr ;
Evaluación: expr → pelotas + paseos * 2

Respeta precedencia de operadores:
  1. paseos * 2 = 4 * 2 = 8
  2. pelotas + 8 = 12 + 8 = 20

declare_variable_checked("total", TYPE_INT, make_int(20))

Tabla de símbolos:
  pelotas: INT = 12
  paseos: INT = 4
  total: INT = 20
```

**Paso 4: Declarar nombre (string)**
```
Parser: statement → CORREA ID = expr ;
Evaluación: expr = "Max"

Tabla de símbolos:
  pelotas: INT = 12
  paseos: INT = 4
  total: INT = 20
  nombre: STRING = "Max"
```

**Paso 5: Concatenar strings**
```
Parser: statement → CORREA ID = expr ;
Evaluación: expr → "Golden: " + nombre

add_values_checked(STRING("Golden: "), STRING("Max"))
Resultado: STRING("Golden: Max")

Tabla de símbolos:
  ...
  texto: STRING = "Golden: Max"
```

**Paso 6: Imprimir texto**
```
Parser: statement → LADRA (expr) ;
Evaluación: expr = texto

print_value_checked(texto)
→ Salida estándar: Golden: Max
```

**Paso 7: Imprimir total**
```
Parser: statement → LADRA (expr) ;
Evaluación: expr = total

print_value_checked(total)
→ Salida estándar: 20
```

**Paso 8: Reasignar pelotas**
```
Parser: statement → ID = expr ;
Evaluación: expr → pelotas - 2

sub_values_checked(12, 2) = 10
assign_symbol("pelotas", make_int(10))

Tabla de símbolos:
  pelotas: INT = 10  ← Actualizado
  paseos: INT = 4
  total: INT = 20
  nombre: STRING = "Max"
  texto: STRING = "Golden: Max"
```

**Paso 9: Imprimir pelotas actualizado**
```
Parser: statement → LADRA (expr) ;
Evaluación: expr = pelotas

print_value_checked(pelotas)
→ Salida estándar: 10
```

### Resumen de Flujo

```
LEXER (lexer.l)
   ↓ Tokenización
PARSER (parser.y)
   ↓ Análisis sintáctico
PARSER_HELPER (parser_helper.c)
   ↓ Operaciones aritméticas y validaciones
SYMBOLS (symbols.c)
   ↓ Almacenamiento y recuperación de variables
SALIDA ESTÁNDAR

Resultado:
  Golden: Max
  20
  10
```

---

## 10. Ejemplos Complejos

### Ejemplo 1: Sistema de Cuidado de Golden Retriever

```
// ============================================
// PROGRAMA: Control de Actividades de Golden
// Descripción: Calcula actividades diarias
// ============================================

/*
  Variables del perro Max
  Almacenan información personal y física
*/
CORREA nombre = "Max";
CORREA raza = "Golden Retriever";

// Datos físicos y biológicos
HUESO edad = 4;
HUESO peso_kg = 30;
HUESO energia_nivel = 9;

// Rutina diaria del perro
HUESO paseos_por_dia = 5;
HUESO minutos_por_paseo = 20;

/*
  Cálculos derivados
  Se basan en los datos base anteriores
*/
HUESO minutos_totales = paseos_por_dia * minutos_por_paseo;

// Mostrar información inicial
LADRA(nombre);              // Salida: Max
LADRA(raza);                // Salida: Golden Retriever
LADRA(minutos_totales);     // Salida: 100

// Simulación: El perro envejece 2 años
edad = edad + 2;
energia_nivel = energia_nivel - 3;
paseos_por_dia = paseos_por_dia - 1;

// Recalcular
minutos_totales = paseos_por_dia * minutos_por_paseo;

// Mostrar estado final
LADRA(edad);             // Salida: 6
LADRA(energia_nivel);    // Salida: 6
LADRA(minutos_totales);  // Salida: 80
```

### Ejemplo 2: Cálculos de Actividades

```
// Programa: Cálculo de energía gastada en actividades

// Datos base
HUESO paseos_semana = 5;
HUESO minutos_paseo = 30;
HUESO calorias_por_minuto = 5;

// Cálculos en cadena
HUESO minutos_totales = paseos_semana * minutos_paseo;
HUESO calorias_quemadas = minutos_totales * calorias_por_minuto;

// Información del perro
CORREA actividad = "Paseos";
CORREA periodo = " por semana";
CORREA resumen = actividad + periodo;

// Mostrar resultados
LADRA(resumen);           // Salida: Paseos por semana
LADRA(minutos_totales);   // Salida: 150
LADRA(calorias_quemadas); // Salida: 750
```

### Ejemplo 3: Matemática Compleja con Precedencia

```
// Programa: Cálculos matemáticos con precedencia de operadores

HUESO a = 10;
HUESO b = 20;
HUESO c = 3;
HUESO d = 4;

// Ejemplo 1: Precedencia básica
HUESO r1 = a + b * c;          // 10 + (20 * 3) = 70
LADRA(r1);  // Salida: 70

// Ejemplo 2: Con paréntesis
HUESO r2 = (a + b) * c;        // (10 + 20) * 3 = 90
LADRA(r2);  // Salida: 90

// Ejemplo 3: Múltiples operaciones
HUESO r3 = a * b / c + d;      // (10 * 20 / 3) + 4 = 66 + 4 = 70
LADRA(r3);  // Salida: 70

// Ejemplo 4: División entera
HUESO r4 = b / c;              // 20 / 3 = 6 (no 6.67)
LADRA(r4);  // Salida: 6

// Ejemplo 5: Expresión compleja
HUESO r5 = a + b / c - d * 2;  // 10 + (20/3) - (4*2) = 10 + 6 - 8 = 8
LADRA(r5);  // Salida: 8
```

---

## 11. Validaciones de Tipo

GoldenRetrieverLang implementa validación de tipos en tiempo de compilación/análisis.

### Validaciones de Declaración

```
✓ HUESO x = 5;           // Correcto: 5 es entero
✗ HUESO y = "text";      // ERROR: "text" no es entero

✓ CORREA s = "Max";      // Correcto: "Max" es string
✗ CORREA t = 42;         // ERROR: 42 no es string
```

### Validaciones de Asignación

```
HUESO a = 1;
✗ a = "oops";    // ERROR: no se puede reasignar string a INT

CORREA s = "Max";
✗ s = 42;        // ERROR: no se puede reasignar entero a STRING
```

### Validaciones de Operaciones Aritméticas

```
// Suma
✓ 10 + 5              // OK: 15
✓ "a" + "b"           // OK: "ab" (concatenación)
✗ 10 + "a"            // ERROR: tipos incompatibles

// Resta (solo enteros)
✓ 10 - 5              // OK: 5
✗ "a" - "b"           // ERROR: resta no permitida para strings

// Multiplicación (solo enteros)
✓ 10 * 5              // OK: 50
✗ "a" * "b"           // ERROR: multiplicación no permitida para strings

// División (solo enteros)
✓ 10 / 5              // OK: 2
✗ 10 / 0              // ERROR: división entre cero
✗ "a" / "b"           // ERROR: división no permitida para strings
```

### Validaciones de Variables

```
✓ HUESO x = 5;
  x = 10;              // OK: reasignación validada

✗ x = "oops";         // ERROR: tipo incompatible

✗ HUESO x = 5;
  HUESO x = 10;        // ERROR: variable duplicada

✗ y = 5;              // ERROR: variable no declarada
```

### Tabla de Validaciones

| Operación | Validación | Resultado |
|-----------|------------|----------|
| INT + INT | ✓ | INT |
| STRING + STRING | ✓ | STRING |
| INT + STRING | ✗ | ERROR |
| INT - INT | ✓ | INT |
| INT - STRING | ✗ | ERROR |
| INT * INT | ✓ | INT |
| INT * STRING | ✗ | ERROR |
| INT / INT | ✓ | INT |
| INT / 0 | ✗ | ERROR |
| INT / STRING | ✗ | ERROR |

---

## Conclusión

GoldenRetrieverLang es un lenguaje académico completo que implementa:

- ✓ **Análisis léxico** con Flex
- ✓ **Análisis sintáctico** con Bison
- ✓ **Tabla de símbolos** con gestión de variables
- ✓ **Operaciones aritméticas** con precedencia correcta
- ✓ **Validación de tipos** en tiempo de análisis
- ✓ **Manejo de memoria** y limpieza de recursos
- ✓ **Comentarios** de línea y bloque
- ✓ **Impresión** en pantalla

Su arquitectura modular en C permite comprender todos los componentes de un compilador/intérprete real.


### Ejemplo 4: Gestión de Inventario de Juguetes

```
// Programa: Sistema de inventario para perro Golden Retriever
// Descripción: Rastreo de juguetes, consumables y accesorios

// ===== INFORMACIÓN DEL PERRO =====
CORREA nombre_perro = "Duke";
CORREA color = "Golden";
CORREA tipo_pelaje = "Largo";

// ===== INVENTARIO INICIAL =====
HUESO pelotas = 15;
HUESO huesos = 8;
HUESO juguetes_mordida = 5;
HUESO camas = 2;

// ===== ESTADOS ACTUALES =====
HUESO comidas_consumidas = 0;
HUESO agua_consumida = 0;

// ===== CÁLCULOS DE CONSUMO =====
HUESO dias_semana = 7;
HUESO comidas_al_dia = 2;
HUESO comidas_semana = dias_semana * comidas_al_dia;

HUESO paseos_diarios = 5;
HUESO duracion_paseo = 20;
HUESO minutos_ejercicio = paseos_diarios * duracion_paseo;

// ===== DESGASTE ESTIMADO =====
HUESO pelotas_perdidas_mes = 3;
HUESO huesos_comidos_mes = 2;

// ===== INVENTARIO PROYECTADO =====
HUESO pelotas_final = pelotas - pelotas_perdidas_mes;
HUESO huesos_final = huesos - huesos_comidos_mes;

// ===== INFORMACIÓN DESCRIPTIVA =====
CORREA descripcion1 = "Duke es color " + color;
CORREA descripcion2 = "Tiene pelaje " + tipo_pelaje;

// ===== SALIDAS =====
LADRA(descripcion1);           // Salida: Duke es color Golden
LADRA(descripcion2);           // Salida: Tiene pelaje Largo

LADRA(pelotas);                // Salida: 15
LADRA(huesos);                 // Salida: 8
LADRA(juguetes_mordida);       // Salida: 5

LADRA(comidas_semana);         // Salida: 14
LADRA(minutos_ejercicio);      // Salida: 100

LADRA(pelotas_final);          // Salida: 12
LADRA(huesos_final);           // Salida: 6
```

### Ejemplo 5: Cálculo de Calorías y Nutrición

```
// Programa: Cálculo de requerimientos nutricionales para Golden Retriever

// ===== PARÁMETROS DEL PERRO =====
HUESO peso_kg = 30;
HUESO edad_anos = 5;
HUESO nivel_actividad = 8;  // Escala 1-10

// ===== CONSTANTES DE CÁLCULO =====
HUESO calorias_base_por_kg = 30;  // Calorias base diarias por kg
HUESO factor_actividad = 2;        // Multiplicador por actividad
HUESO proteina_porcentaje = 25;    // 25% de las calorías
HUESO grasa_porcentaje = 15;       // 15% de las calorías
HUESO carbohidrato_porcentaje = 50; // 50% de las calorías

// ===== CÁLCULOS PRINCIPALES =====
HUESO calorias_base = peso_kg * calorias_base_por_kg;
HUESO calorias_ajustadas = calorias_base * nivel_actividad / 10;

// ===== DISTRIBUCIÓN NUTRICIONAL =====
HUESO calorias_proteina = calorias_ajustadas * proteina_porcentaje / 100;
HUESO calorias_grasa = calorias_ajustadas * grasa_porcentaje / 100;
HUESO calorias_carbohidrato = calorias_ajustadas * carbohidrato_porcentaje / 100;

// ===== CONVERSIÓN A GRAMOS =====
HUESO gramos_proteina = calorias_proteina / 4;     // 1g proteína = 4 calorías
HUESO gramos_grasa = calorias_grasa / 9;           // 1g grasa = 9 calorías
HUESO gramos_carbohidrato = calorias_carbohidrato / 4; // 1g carbohidrato = 4 calorías

// ===== PORCIONAMIENTO DIARIO =====
HUESO porciones_diarias = 2;
HUESO gramos_proteina_porcion = gramos_proteina / porciones_diarias;
HUESO gramos_grasa_porcion = gramos_grasa / porciones_diarias;

// ===== SALIDAS DE RESULTADOS =====
LADRA(calorias_base);               // Salida: 900
LADRA(calorias_ajustadas);          // Salida: 720

LADRA(gramos_proteina);             // Salida: 45
LADRA(gramos_grasa);                // Salida: 12
LADRA(gramos_carbohidrato);         // Salida: 90

LADRA(gramos_proteina_porcion);     // Salida: 22
LADRA(gramos_grasa_porcion);        // Salida: 6
```

### Ejemplo 6: Seguimiento de Salud y Vacunas

```
// Programa: Calendario de salud y vacunación de Golden Retriever

// ===== INFORMACIÓN GENERAL =====
CORREA nombre = "Charlie";
CORREA raza = "Golden Retriever";
CORREA color_pelaje = "Crema";

// ===== EDAD =====
HUESO edad_meses = 36;  // 3 años = 36 meses

// ===== CÁLCULOS DE EDAD =====
HUESO anos_vida = edad_meses / 12;
HUESO meses_restantes = edad_meses - (anos_vida * 12);

// ===== HISTORIAL DE VACUNAS =====
HUESO vacunas_aplicadas = 5;
HUESO dosis_rabia = 2;
HUESO dosis_parvovirus = 3;
HUESO dosis_distemper = 2;

// ===== CÁLCULOS DE PRÓXIMAS VACUNAS =====
HUESO dias_entre_refuerzos = 365;  // Un año
HUESO proximas_vacunas_en_dias = dias_entre_refuerzos - 30;

// ===== VISITAS VETERINARIAS =====
HUESO visitas_ano = 2;
HUESO visitas_totales = anos_vida * visitas_ano;

// ===== DATOS DE SALUD =====
HUESO peso_kg = 32;
HUESO peso_ideal_min = 30;
HUESO peso_ideal_max = 35;
HUESO diferencia_peso = peso_kg - peso_ideal_min;

// ===== SALIDAS =====
LADRA(nombre);                      // Salida: Charlie
LADRA(raza);                        // Salida: Golden Retriever

LADRA(edad_meses);                  // Salida: 36
LADRA(anos_vida);                   // Salida: 3

LADRA(vacunas_aplicadas);           // Salida: 5
LADRA(dosis_rabia);                 // Salida: 2
LADRA(visitas_totales);             // Salida: 6

LADRA(peso_kg);                     // Salida: 32
LADRA(diferencia_peso);             // Salida: 2
```

### Ejemplo 7: Simulación de Entrenamiento Progresivo

```
// Programa: Plan de entrenamiento progresivo para Golden Retriever joven

// ===== SEMANA 1: INICIO DEL ENTRENAMIENTO =====
HUESO semana = 1;
HUESO comandos_ensenados_s1 = 3;     // Siéntate, Ven, Quieto
HUESO horas_practica_s1 = semana * 2; // 2 horas

// ===== SEMANA 2: PROGRESO =====
HUESO comandos_ensenados_s2 = 5;     // Agrega: Acuesta, Pata
HUESO horas_practica_s2 = 4;         // 4 horas

// ===== SEMANA 3: MÁS AVANZADO =====
HUESO comandos_ensenados_s3 = 8;
HUESO horas_practica_s3 = 6;

// ===== SEMANA 4: CONSOLIDACIÓN =====
HUESO comandos_ensenados_s4 = 10;
HUESO horas_practica_s4 = 8;

// ===== CÁLCULOS ACUMULATIVOS =====
HUESO total_comandos = comandos_ensenados_s1 + comandos_ensenados_s2 + comandos_ensenados_s3 + comandos_ensenados_s4;
HUESO total_horas_practica = horas_practica_s1 + horas_practica_s2 + horas_practica_s3 + horas_practica_s4;

// ===== MEJORA PROGRESIVA =====
HUESO mejora_s2 = comandos_ensenados_s2 - comandos_ensenados_s1;
HUESO mejora_s3 = comandos_ensenados_s3 - comandos_ensenados_s2;
HUESO mejora_s4 = comandos_ensenados_s4 - comandos_ensenados_s3;

// ===== INFORMACIÓN DESCRIPTIVA =====
CORREA fase1 = "Semana 1: Introduccion";
CORREA fase2 = "Semana 2: Consolidacion";
CORREA fase3 = "Semana 3: Expansion";
CORREA fase4 = "Semana 4: Maestria";

CORREA titulo = "PLAN DE ENTRENAMIENTO - GOLDEN RETRIEVER";

// ===== SALIDAS =====
LADRA(titulo);                      // Salida: PLAN DE ENTRENAMIENTO - GOLDEN RETRIEVER

LADRA(fase1);                       // Salida: Semana 1: Introduccion
LADRA(comandos_ensenados_s1);       // Salida: 3
LADRA(horas_practica_s1);           // Salida: 2

LADRA(fase2);                       // Salida: Semana 2: Consolidacion
LADRA(comandos_ensenados_s2);       // Salida: 5
LADRA(mejora_s2);                   // Salida: 2

LADRA(fase3);                       // Salida: Semana 3: Expansion
LADRA(comandos_ensenados_s3);       // Salida: 8
LADRA(mejora_s3);                   // Salida: 3

LADRA(fase4);                       // Salida: Semana 4: Maestria
LADRA(comandos_ensenados_s4);       // Salida: 10
LADRA(mejora_s4);                   // Salida: 2

LADRA(total_comandos);              // Salida: 26
LADRA(total_horas_practica);        // Salida: 20
```

### Ejemplo 8: Análisis de Gastos en Cuidado

```
// Programa: Cálculo de gastos mensuales en cuidado de Golden Retriever

// ===== NOMBRE DEL PERRO Y UBICACIÓN =====
CORREA nombre = "Bailey";
CORREA ciudad = "San Francisco";

// ===== GASTOS MENSUALES =====
HUESO costo_comida = 150;          // Comida de calidad
HUESO costo_veterinario = 80;      // Consultas y checkups
HUESO costo_aseo = 60;             // Baño y corte
HUESO costo_juguetes = 40;         // Juguetes y accesorios
HUESO costo_medicinas = 30;        // Medicamentos y vitaminas
HUESO costo_entrenamiento = 100;   // Clases de entrenamiento

// ===== CÁLCULOS TOTALES =====
HUESO gasto_total_mes = costo_comida + costo_veterinario + costo_aseo + costo_juguetes + costo_medicinas + costo_entrenamiento;

// ===== PROYECCIONES =====
HUESO meses_ano = 12;
HUESO gasto_anual = gasto_total_mes * meses_ano;

HUESO semanas_mes = 4;
HUESO gasto_semanal = gasto_total_mes / semanas_mes;

// ===== DESGLOSE PORCENTUAL =====
HUESO porcentaje_comida = costo_comida * 100 / gasto_total_mes;
HUESO porcentaje_veterinario = costo_veterinario * 100 / gasto_total_mes;
HUESO porcentaje_aseo = costo_aseo * 100 / gasto_total_mes;

// ===== INFORMACIÓN DESCRIPTIVA =====
CORREA resumen_inicio = "Analisis de gastos para ";
CORREA resumen_largo = resumen_inicio + nombre;
CORREA ubicacion_texto = " en " + ciudad;

CORREA detalle_comida = "Comida: Un rubro importante";
CORREA detalle_vet = "Veterinario: Esencial";

// ===== SALIDAS RESUMEN =====
LADRA(nombre);                     // Salida: Bailey
LADRA(ciudad);                     // Salida: San Francisco

LADRA(detalle_comida);             // Salida: Comida: Un rubro importante
LADRA(costo_comida);               // Salida: 150

LADRA(detalle_vet);                // Salida: Veterinario: Esencial
LADRA(costo_veterinario);          // Salida: 80

// ===== SALIDAS CÁLCULOS =====
LADRA(gasto_total_mes);            // Salida: 460
LADRA(gasto_anual);                // Salida: 5520
LADRA(gasto_semanal);              // Salida: 115

// ===== DESGLOSE PORCENTUAL =====
LADRA(porcentaje_comida);          // Salida: 32
LADRA(porcentaje_veterinario);     // Salida: 17
LADRA(porcentaje_aseo);            // Salida: 13

// ===== COMPARATIVA =====
HUESO presupuesto_disponible = 500;
HUESO diferencia_presupuesto = presupuesto_disponible - gasto_total_mes;

LADRA(presupuesto_disponible);     // Salida: 500
LADRA(diferencia_presupuesto);     // Salida: 40
```